# 2030 / 2040 / 2050 最优解场站位置可视化

对应论文：*Globally Interconnected Solar-Wind System Addresses Future Electricity Demands*

**本 notebook 目标：**
1. 从单目标成本最小化优化结果 (`.mat`) 提取 2030/2040/2050 年的最优解场站位置
2. 可视化三个年代的光伏和风电场站部署
3. 导出场站坐标到 CSV

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import scipy.io
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# ── 字体 ──
font_path = "/data4/yanxiaokai/SourceHanSansSC-Normal.otf"
fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = [font_name]
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
    }
)

# ── 路径配置 ──
# 修改此处以切换 SSP 情景或结果目录
SSP_DIR = "Optimization_ssp245"
RESULT_SUBDIR = "result_20260604"  # None 表示使用 results/ 目录

if RESULT_SUBDIR:
    RESULT_DIR = os.path.join(SSP_DIR, "results", RESULT_SUBDIR)
else:
    RESULT_DIR = os.path.join(SSP_DIR, "results")

# Sel.mat 文件名（与 convert_h5_to_sel.m 输出一致）
SEL_FILES = {
    2050: os.path.join(RESULT_DIR, "Opt_SC_2050_Sel.mat"),
    2040: os.path.join(RESULT_DIR, "Opt_SC_2040_Sel.mat"),
    2030: os.path.join(RESULT_DIR, "Opt_SA_2030_Sel.mat"),
}

REGION_NAMES = [
    "Northern America",
    "Caribbean",
    "Central America",
    "Southern America",
    "Northern Europe",
    "Western Europe",
    "Southern Europe",
    "Eastern Europe",
    "Central Asia",
    "Eastern Asia",
    "Southern Asia",
    "Southeastern Asia",
    "Western Asia",
    "Australia-NZ",
    "Melanesia",
    "Northern Africa",
    "Western Africa",
    "Middle Africa",
    "Eastern Africa",
    "Southern Africa",
]

print("环境配置完成")
print(f"结果目录: {RESULT_DIR}")
for year, fpath in SEL_FILES.items():
    exists = os.path.isfile(fpath)
    print(f"  {year}: {fpath} {'[OK]' if exists else '[MISSING]'}")

In [2]:
# ── 工具函数 ──


def setup_basemap(ax):
    """绘制地图底图（海岸线、国界、网格线）."""
    ax.set_global()
    ax.add_feature(cfeature.LAND, color="#f5f5f0", zorder=0)
    ax.add_feature(cfeature.OCEAN, color="#d1e8f0", zorder=0)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, color="#888", zorder=1)
    ax.add_feature(cfeature.BORDERS, linewidth=0.3, color="#aaa", zorder=1)
    ax.gridlines(linewidth=0.3, color="gray", alpha=0.4)


def get_candidates_from_opt(opt_file):
    """
    从上层优化结果 .mat 文件中提取选中格网的经纬度坐标.

    Opt_*.mat 包含 opt_solar / opt_wind (180×360 二值栅格),
    非零位置即该层最优解选中的场站。

    嵌套逻辑:
      2050 场站 = find(Opt_SG_2050_Sel.mat 的 opt_solar/wind==1) → 同时也是 SC_2040 候选
      2040 场站 = find(Opt_SC_2040_Sel.mat 的 opt_solar/wind==1) → 同时也是 SA_2030 候选
      2030 场站 = 从 .h5 解#5 中提取 (候选 = 2040 场站)

    MATLAB find() 返回 column-major 索引, 与 .h5 决策变量顺序一致。

    Returns: (n_solar, slon, slat, n_wind, wlon, wlat)
    """
    mat = scipy.io.loadmat(opt_file)
    opt_solar = mat["opt_solar"]  # (180, 360)
    opt_wind = mat["opt_wind"]  # (180, 360)
    nrows = opt_solar.shape[0]  # 180

    # MATLAB find(X) = np.nonzero(X.ravel(order='F'))
    sflat = np.nonzero(opt_solar.ravel(order="F"))[0]
    wflat = np.nonzero(opt_wind.ravel(order="F"))[0]

    srows, scols = sflat % nrows, sflat // nrows
    wrows, wcols = wflat % nrows, wflat // nrows

    # 180×360 grid (1°×1°): row→lat, col→lon
    slon, slat = -179.5 + scols, 89.5 - srows
    wlon, wlat = -179.5 + wcols, 89.5 - wrows

    return len(sflat), slon, slat, len(wflat), wlon, wlat


print("工具函数定义完成")

工具函数定义完成


---
## 场站坐标提取与可视化

In [ ]:
# ── 提取各年选中场站坐标 ──
# 单目标优化: .mat 文件中的 opt_solar/opt_wind 直接给出唯一最优解

year_colors = {2030: "#c501ff", 2040: "#00ffc5", 2050: "#d48a8b"}
interconnection_labels = {2050: "S-G (Global)", 2040: "S-C (Continental)", 2030: "S-A (Adjacent)"}

scenarios = {}
for year in [2050, 2040, 2030]:
    ns, slon, slat, nw, wlon, wlat = get_candidates_from_opt(SEL_FILES[year])
    scenarios[year] = dict(
        slon=slon, slat=slat, wlon=wlon, wlat=wlat,
        n_solar=ns, n_wind=nw,
    )
    print(f"  {year} ({interconnection_labels[year]}): 光伏站={ns:,}  风电站={nw:,}")

# ── 绘图 ──
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(16, 14),
    subplot_kw={"projection": ccrs.Robinson()},
)
setup_basemap(ax1)
setup_basemap(ax2)

for year in [2050, 2040, 2030]:
    sc = scenarios[year]
    ax1.scatter(
        sc["slon"], sc["slat"], s=3, c=year_colors[year],
        transform=ccrs.PlateCarree(),
        label=f"{year} ({sc['n_solar']:,})",
        rasterized=True,
    )
    ax2.scatter(
        sc["wlon"], sc["wlat"], s=3, c=year_colors[year],
        transform=ccrs.PlateCarree(),
        label=f"{year} ({sc['n_wind']:,})",
        rasterized=True,
    )

for ax, title in [
    (ax1, "2030 / 2040 / 2050 Optimal Solution — Solar Stations"),
    (ax2, "2030 / 2040 / 2050 Optimal Solution — Wind Stations"),
]:
    ax.legend(loc="lower left", fontsize=11, markerscale=5, framealpha=0.9, edgecolor="#888")
    ax.set_title(title, fontsize=13, pad=10)

plt.tight_layout()
os.makedirs("output", exist_ok=True)
plt.savefig("output/solar_wind_optimization_scenarios.png", dpi=300)
plt.show()

In [ ]:
# 保存 2030、2040、 2050 场站位置到 csv 文件
import pandas as pd

records = []
for year in [2030, 2040, 2050]:
    sc = scenarios[year]
    for lon, lat in zip(sc["slon"], sc["slat"]):
        records.append({"year": year, "type": "solar", "lon": lon, "lat": lat})
    for lon, lat in zip(sc["wlon"], sc["wlat"]):
        records.append({"year": year, "type": "wind", "lon": lon, "lat": lat})

df = pd.DataFrame(records)
df.to_csv("output/optimal_stations.csv", index=False)
print(f"已保存所有选中场站坐标到 output/optimal_stations.csv (共 {len(df):,} 条记录)")